In [ ]:
# Импорт необходимых библиотек

import warnings
warnings.filterwarnings('ignore')

import os
import sys
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data, paths, loop
from src import threshold as threshold_mod

In [ ]:
# Конфигурация

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = Path(r'path')        # датасет: train.csv + img/ + mask/ + src/
TEST_DIR = Path(r'path')        # тестовая папка с test.csv
CKPT_MAIN = Path(r'path')       # главная модель (кроповая -> оценка окнами)
CKPT_SECOND = Path(r'path')     # вторая модель (ресайзная -> оценка целиком)

IMG_SIZE = 576
BATCH_SIZE = 8
NUM_WORKERS = 0     # в ноутбуке — 0 (Jupyter + Windows)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# Модель

def build_model():
    return smp.Unet(
        encoder_name='timm-efficientnet-b3',
        encoder_weights=None,   # оценочный режим: imagenet-веса не нужны
        in_channels=5,
        classes=1,
        activation=None,
        aux_params=dict(pooling='avg', dropout=0.2, classes=1),
    )

In [ ]:
# Данные

rows = data.read_train_csv(DATA_DIR / 'train.csv', DATA_DIR, max_rows=None, seed=42)
flags = data.compute_has_mask_flags(rows)
tr_rows, tr_flags, val_rows, val_flags = data.stratified_split(rows, flags, 0.1, 42)
val_rows = val_rows + data.orgl_clean_rows(val_rows)

gt_maps = [(data.load_gt_mask_for_row(r) > 0.5).astype(np.uint8) for r in val_rows]
print(f'val: {len(val_rows)} строк (чистых: {sum(1 for r in val_rows if r["gt"] is None)})')

In [ ]:
# Инференс. TTA, слайдинг виндоу, сбор prob карт

@torch.no_grad()
def tta_probs(net, images):

    def seg_aux(x):
        seg, aux = net(x)
        return torch.sigmoid(seg), torch.sigmoid(aux)

    seg_acc, aux_acc = seg_aux(images)
    s, a = seg_aux(torch.flip(images, dims=[3]))
    seg_acc = seg_acc + torch.flip(s, dims=[3]); aux_acc = aux_acc + a
    s, a = seg_aux(torch.flip(images, dims=[2]))
    seg_acc = seg_acc + torch.flip(s, dims=[2]); aux_acc = aux_acc + a

    return (seg_acc / 3.0).cpu().numpy()[:, 0], (aux_acc / 3.0).cpu().numpy().reshape(-1)


def probs_to_native(prob, orig_h, orig_w):
    return cv2.resize(prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)


def predict_windows(net, row, crop_size=576, overlap=0.33):
    img = data._load_rgb(row['chng'])
    h, w = img.shape[:2]
    step = int(crop_size * (1 - overlap))

    ys_list = list(range(0, max(1, h - crop_size + 1), step))
    xs_list = list(range(0, max(1, w - crop_size + 1), step))
    if ys_list[-1] + crop_size < h: ys_list.append(h - crop_size)
    if xs_list[-1] + crop_size < w: xs_list.append(w - crop_size)
    ys_list = [max(0, y) for y in ys_list]
    xs_list = [max(0, x) for x in xs_list]

    acc = np.zeros((h, w), dtype=np.float32)
    cnt = np.zeros((h, w), dtype=np.float32)
    aux_acc, aux_n = 0.0, 0

    for ys in ys_list:
        for xs in xs_list:
            y2, x2 = min(ys + crop_size, h), min(xs + crop_size, w)
            crop = img[ys:y2, xs:x2]
            ch, cw = crop.shape[:2]
            pad_h, pad_w = crop_size - ch, crop_size - cw
            if pad_h > 0 or pad_w > 0:
                crop = np.pad(crop, ((0, pad_h), (0, pad_w), (0, 0)))
            ela = cv2.resize(data.forensics.compute_ela(crop), (crop_size, crop_size),
                             interpolation=cv2.INTER_LINEAR)
            hp = cv2.resize(data.forensics.compute_highpass(crop), (crop_size, crop_size),
                            interpolation=cv2.INTER_LINEAR)
            stacked = data.stack_channels(crop, ela, hp)
            window = torch.from_numpy(stacked.transpose(2, 0, 1))[None].float().to(DEVICE)

            probs, auxs = tta_probs(net, window)
            acc[ys:y2, xs:x2] += probs[0][:ch, :cw]
            cnt[ys:y2, xs:x2] += 1.0
            aux_acc += float(auxs[0]); aux_n += 1

    return acc / np.maximum(cnt, 1), aux_acc / max(aux_n, 1)


def collect_windows(ckpt_path, rows_list):

    """Кроповая модель -> prob-карты слайдинг-виндоу (долго на полном объёме)."""

    m = build_model().to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    pmaps, aprobs = [], []
    for row in tqdm(rows_list, desc=f'windows {Path(ckpt_path).name}'):
        pmap, ap = predict_windows(m, row)
        pmaps.append(pmap.astype(np.float16)); aprobs.append(ap)
    del m
    torch.cuda.empty_cache()
    return pmaps, aprobs


def collect_full(ckpt_path, rows_list):

    """Ресайзная модель -> prob-карты целых картинок."""

    m = build_model().to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    ds = data.SegDataset(rows_list, IMG_SIZE, train=False, use_forensics=True)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    pmaps, aprobs = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'full {Path(ckpt_path).name}'):
            probs, auxs = tta_probs(m, batch['image'].to(DEVICE))
            for i in range(probs.shape[0]):
                meta = {k: int(batch[k][i]) for k in ('orig_h', 'orig_w')}
                pmaps.append(probs_to_native(probs[i], meta['orig_h'], meta['orig_w']))
                aprobs.append(float(auxs[i]))
    del m
    torch.cuda.empty_cache()
    return pmaps, aprobs

In [ ]:
# Оценка соло обеих моделей + ансамбль
# Для быстрой оценки замените val_rows на val_rows[:500] во всех трёх вызовах ниже

p_main, a_main = collect_windows(CKPT_MAIN, val_rows)

solo_main = threshold_mod.grid_search_threshold(p_main, gt_maps, a_main)

p_second, a_second = collect_full(CKPT_SECOND, val_rows)
solo_second = threshold_mod.grid_search_threshold(p_second, gt_maps, a_second)

ens = [((x.astype(np.float32) + y.astype(np.float32)) / 2).astype(np.float16)
       for x, y in zip(p_main, p_second)]

ens_aux = [(x + y) / 2 for x, y in zip(a_main, a_second)]

chosen_pair = threshold_mod.grid_search_threshold(ens, gt_maps, ens_aux)

In [ ]:
# Сабмит
# По умолчанию ансамбль обеих моделей. Если победило соло главной, то используйте p_main_t / a_main_t напрямую вместо усреднения

test_rows = data.read_test_csv(TEST_DIR / 'test.csv', TEST_DIR)

p_main_t, a_main_t = collect_windows(CKPT_MAIN, test_rows)
p_second_t, a_second_t = collect_full(CKPT_SECOND, test_rows)

ens_t = [((x.astype(np.float32) + y.astype(np.float32)) / 2).astype(np.float16)
         for x, y in zip(p_main_t, p_second_t)]
ens_aux_t = [(x + y) / 2 for x, y in zip(a_main_t, a_second_t)]

PRED_DIR = Path('predictions')
PRED_DIR.mkdir(exist_ok=True)
sub_rows = []

for row, pmap, ap in tqdm(list(zip(test_rows, ens_t, ens_aux_t)), desc='masks'):
    binary = threshold_mod.postprocess_mask(
        pmap.astype(np.float32),
        chosen_pair['prob_threshold'], chosen_pair['min_area'],
        chosen_pair['reject_threshold'], ap, chosen_pair.get('gate_threshold', 0.0))
    png = (binary * 255).astype(np.uint8)

    assert png.shape == pmap.shape
    assert set(np.unique(png).tolist()).issubset({0, 255})

    name = Path(row['chng_rel']).with_suffix('').as_posix().replace('/', '__') + '_pred.png'
    cv2.imwrite(str(PRED_DIR / name), png)
    sub_rows.append({'img_path': row['chng_rel'], 'prediction_path': f'{PRED_DIR.name}/{name}'})

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['img_path', 'prediction_path'])
    w.writeheader()
    w.writerows(sub_rows)